# 07 時間序列與預測 — 參考解答

松柏護理之家退伍軍人症群聚事件時間序列練習的完整解答。

In [ ]:
# Google Colab setup -- 若在本機執行可跳過此 cell
import sys
import os
if 'google.colab' in sys.modules:
    !git clone https://github.com/ancientsky/python4epi.git /content/python4epi 2>/dev/null || true
    os.chdir('/content/python4epi')
    !pip install -q -e .

In [ ]:
import pathlib

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
from sklearn.metrics import mean_absolute_error

# -- CJK font setup (避免中文標籤顯示為方框) --
# 掃描系統字型目錄，顯式註冊 CJK 字型（比依賴快取更可靠）
for _font_dir in map(pathlib.Path, ["/usr/share/fonts", "/usr/local/share/fonts"]):
    if _font_dir.exists():
        for _fp in sorted(_font_dir.rglob("*")):
            if _fp.suffix.lower() in {".ttf", ".ttc", ".otf"} and (
                "CJK" in _fp.name or "WenQuanYi" in _fp.name or "wqy" in _fp.name
            ):
                try:
                    fm.fontManager.addfont(str(_fp))
                except Exception:
                    pass

plt.rcParams["font.sans-serif"] = [
    "Noto Sans CJK TC", "Noto Sans CJK SC", "Noto Sans CJK JP",
    "Noto Sans TC", "Microsoft JhengHei",
    "WenQuanYi Zen Hei", "SimHei", "Arial Unicode MS",
    "Heiti TC", "DejaVu Sans",
]
plt.rcParams["axes.unicode_minus"] = False
plt.style.use("ggplot")
plt.rcParams["figure.dpi"] = 150

df = pd.read_csv("data/synthetic/legionella_outbreak.csv")
df["symptom_onset_date"] = pd.to_datetime(df["symptom_onset_date"], errors="coerce")
df["hospitalization_date"] = pd.to_datetime(df["hospitalization_date"], errors="coerce")
df["infected"] = (df["clinical_severity"] != "not_ill").astype(int)
cases = df[df["infected"] == 1]

## 題目 1：建立每日住院數序列

In [ ]:
import matplotlib.dates as mdates

# 每日住院數
hosp_cases = cases[cases["hospitalization_date"].notna()]
hosp_daily = hosp_cases.groupby("hospitalization_date").size()
hosp_daily = hosp_daily.asfreq("D", fill_value=0)
hosp_daily.name = "hospitalizations"

print(f"序列長度：{len(hosp_daily)} 天")
print(f"日期範圍：{hosp_daily.index.min().date()} – {hosp_daily.index.max().date()}")
print(f"住院總數：{hosp_daily.sum()}")

# 加入背景期
date_range = pd.date_range(
    hosp_daily.index.min() - pd.Timedelta(days=3),
    hosp_daily.index.max() + pd.Timedelta(days=1),
)
hosp_plot = hosp_daily.reindex(date_range, fill_value=0)

# 住院曲線 + 5 日滾動平均
rolling_5 = hosp_daily.rolling(window=5, min_periods=1).mean()

fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(
    hosp_plot.index, hosp_plot.values,
    width=1.0,
    color="#e34a33", edgecolor="white", linewidth=0.5,
    alpha=0.7, label="每日住院",
)
ax.plot(rolling_5.index, rolling_5.values, color="navy", linewidth=2,
        label="5 日滾動平均")
ax.set_title(
    "松柏護理之家退伍軍人症每日住院曲線 + 5 日滾動平均，2026 年 1 月",
    fontsize=13, fontweight="bold",
)
ax.set_xlabel("住院日期（Date of Hospitalization）")
ax.set_ylabel("住院人數（Number of Hospitalizations）")

ax.xaxis.set_major_formatter(mdates.DateFormatter("%m/%d"))
ax.xaxis.set_major_locator(mdates.DayLocator(interval=2))
fig.autofmt_xdate(rotation=45, ha="right")

ax.set_ylim(bottom=0)
ax.yaxis.set_major_locator(plt.MaxNLocator(integer=True))
ax.grid(False)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.legend()
plt.tight_layout()
plt.show()

## 題目 2：住院數預測與窗口比較

In [ ]:
# 窗口比較
print("=== 住院數預測 MAE ===")
best_w, best_mae = 3, float("inf")

for w in [3, 5, 7]:
    pred_w = hosp_daily.rolling(window=w).mean().shift(1).dropna()
    actual_w = hosp_daily.loc[pred_w.index]
    mae_w = mean_absolute_error(actual_w, pred_w)
    print(f"  window={w}  MAE={mae_w:.3f}")
    if mae_w < best_mae:
        best_w, best_mae = w, mae_w

print(f"\n→ 最佳窗口：window={best_w}（MAE={best_mae:.3f}）")

# Actual vs Predicted
pred_best = hosp_daily.rolling(window=best_w).mean().shift(1).dropna()
actual_best = hosp_daily.loc[pred_best.index]

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(actual_best.index, actual_best.values, marker="o", markersize=4,
        label="實際住院", color="#e34a33")
ax.plot(pred_best.index, pred_best.values, marker="s", markersize=4,
        label=f"預測（{best_w} 日 MA）", color="navy", linestyle="--")
ax.set_title(
    f"松柏護理之家住院數 Actual vs Predicted（{best_w} 日滾動平均），2026 年 1 月",
    fontsize=13, fontweight="bold",
)
ax.set_xlabel("日期（Date）")
ax.set_ylabel("每日住院數（Number of Hospitalizations）")

ax.xaxis.set_major_formatter(mdates.DateFormatter("%m/%d"))
ax.xaxis.set_major_locator(mdates.DayLocator(interval=2))
fig.autofmt_xdate(rotation=45, ha="right")

ax.set_ylim(bottom=0)
ax.yaxis.set_major_locator(plt.MaxNLocator(integer=True))
ax.grid(False)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.legend()
plt.tight_layout()
plt.show()

## 題目 3（挑戰題）：按嚴重度分組的流行曲線

In [ ]:
# 按嚴重度建立每日發病數序列
severity_levels = ["mild", "moderate", "severe"]
colors = {"mild": "#41b6c4", "moderate": "#fed976", "severe": "#e31a1c"}

# 建立每日序列（所有嚴重度共用日期範圍，含背景期）
all_onset = cases.groupby("symptom_onset_date").size()
all_onset = all_onset.asfreq("D", fill_value=0)

# 加入背景期
date_range = pd.date_range(
    all_onset.index.min() - pd.Timedelta(days=3),
    all_onset.index.max() + pd.Timedelta(days=1),
)

severity_daily = {}
for sev in severity_levels:
    sub = cases[cases["clinical_severity"] == sev]
    s = sub.groupby("symptom_onset_date").size()
    severity_daily[sev] = s.reindex(date_range, fill_value=0)

sev_df = pd.DataFrame(severity_daily)

# Stacked bar chart
fig, ax = plt.subplots(figsize=(10, 4))
bottom = np.zeros(len(sev_df))

for sev in severity_levels:
    ax.bar(
        sev_df.index, sev_df[sev].values, bottom=bottom,
        width=1.0,
        color=colors[sev], edgecolor="white", linewidth=0.5,
        alpha=0.8, label=sev,
    )
    bottom += sev_df[sev].values

ax.set_title(
    "松柏護理之家退伍軍人症流行曲線（按嚴重度分層），2026 年 1 月",
    fontsize=13, fontweight="bold",
)
ax.set_xlabel("發病日期（Date of Symptom Onset）")
ax.set_ylabel("病例數（Number of Cases）")

ax.xaxis.set_major_formatter(mdates.DateFormatter("%m/%d"))
ax.xaxis.set_major_locator(mdates.DayLocator(interval=2))
fig.autofmt_xdate(rotation=45, ha="right")

ax.set_ylim(bottom=0)
ax.yaxis.set_major_locator(plt.MaxNLocator(integer=True))
ax.grid(False)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.legend()
plt.tight_layout()
plt.show()

# 各嚴重度的高峰日
print("=== 各嚴重度高峰日 ===")
for sev in severity_levels:
    peak_date = sev_df[sev].idxmax()
    peak_count = sev_df[sev].max()
    print(f"  {sev:10s}  高峰日 = {peak_date.date()}  當日 {peak_count} 人")

print("\n→ 觀察重症病例是否與輕症同步出現，或有時間延遲")
print("→ 如果重症集中在疫情中期，可能代表暴露劑量較高的住民較晚發病")

### 解讀

- **住院曲線**：住院高峰比發病高峰晚幾天，這個 lag 可用於預測床位需求
- **窗口選擇**：較小窗口（3 日）通常在急性群聚中表現較好，因為病例數變化快
- **嚴重度分層**：如果重症病例集中在某個時間段，可能提示特定暴露事件或高風險族群
- **限制**：滾動平均是最簡單的 baseline 模型，無法捕捉趨勢轉折點；更進階的方法（如 ARIMA）可在此基礎上改進

## 題目 4 解答

In [ ]:
import numpy as np

# --- 資料：腸病毒（Enterovirus）2 年通報 line list ---
rng = np.random.default_rng(701)
start = pd.Timestamp("2024-01-01")
n_days = 730
dates_all = pd.date_range(start, periods=n_days, freq="D")
doy = np.array([d.dayofyear for d in dates_all]) % 366

# 兩個季節高峰：初夏（約 4 月）與開學季（約 9-10 月）
peak1 = np.exp(-0.5 * ((doy - 100) / 22) ** 2)
peak2 = np.exp(-0.5 * ((doy - 270) / 28) ** 2)
weights = 0.05 + peak1 + 0.7 * peak2
probs = weights / weights.sum()

n_cases = 480
idx = rng.choice(n_days, size=n_cases, p=probs)
line_list = pd.DataFrame({
    "case_id": np.arange(1, n_cases + 1),
    "report_date": dates_all[idx],
    "age_group": rng.choice(["<5", "5-9", "10-14"], size=n_cases, p=[0.55, 0.30, 0.15]),
}).sort_values("report_date").reset_index(drop=True)

# 每日 -> 每週通報數（補齊缺週）
daily = line_list.groupby("report_date").size()
daily = daily.asfreq("D", fill_value=0)
weekly = daily.resample("W").sum()
weekly.name = "cases"
print(f"序列長度：{len(weekly)} 週 | 總通報數：{weekly.sum()}")

# 各月份平均週通報數
monthly_avg = weekly.groupby(weekly.index.month).mean().sort_values(ascending=False)
print("\n=== 各月份平均週通報數（前 5 高）===")
print(monthly_avg.head().round(2))

# 週對週變化量
wow_change = (weekly - weekly.shift(1)).dropna()
print(f"\n週對週變化量：平均 {wow_change.mean():+.2f}，最大單週增幅 {wow_change.max():.0f}")

# 每週流行曲線，標出前 8 高峰週
top_weeks = weekly.sort_values(ascending=False).head(8)

fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(weekly.index, weekly.values, width=5, color="#6A9BCC",
       edgecolor="white", alpha=0.75, label="每週通報數")
ax.scatter(top_weeks.index, top_weeks.values, color="#D97757", s=45,
           zorder=5, label="前 8 高峰週")
ax.set_title("腸病毒每週通報數（2024-2025）", fontweight="bold")
ax.set_xlabel("週別（Week）")
ax.set_ylabel("通報數（Number of Reports）")
ax.legend()
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
fig.autofmt_xdate()
plt.tight_layout()
plt.show()

print("\n→ 通報數呈現明顯雙峰季節性：每年 4 月前後（初夏）與 9-10 月（開學季）最高")
print("→ 可能與兒童在校園、安親班等場所群聚接觸增加、腸病毒經糞口與飛沫傳播有關")

## 題目 5 解答

In [ ]:
import numpy as np

# --- 資料：流感（Influenza）120 天通報 line list ---
rng = np.random.default_rng(707)
start = pd.Timestamp("2025-11-01")
n_days = 120
dates_all = pd.date_range(start, periods=n_days, freq="D")
day_idx = np.arange(n_days)

# 單一流感季：中段達到高峰（約第 60 天），加上週末通報偏低的型態
season_shape = np.exp(-0.5 * ((day_idx - 60) / 18) ** 2)
dow_weight = np.where(dates_all.dayofweek < 5, 1.0, 0.45)  # 假日就診/通報較少
weights = (0.08 + season_shape) * dow_weight
probs = weights / weights.sum()

n_cases = 560
idx = rng.choice(n_days, size=n_cases, p=probs)
line_list = pd.DataFrame({
    "case_id": np.arange(1, n_cases + 1),
    "report_date": dates_all[idx],
}).sort_values("report_date").reset_index(drop=True)

daily = line_list.groupby("report_date").size()
daily = daily.asfreq("D", fill_value=0)
daily.name = "cases"
print(f"序列長度：{len(daily)} 天 | 總通報數：{daily.sum()}")

# 平穩性檢定
from statsmodels.tsa.stattools import adfuller
adf_stat, p_value, *_ = adfuller(daily)
print(f"\nADF statistic = {adf_stat:.3f}, p-value = {p_value:.3f}")
print("→ p >= 0.05：序列非平穩（有明顯季節趨勢），需要差分（d, D >= 1）")

# 切訓練 / 測試集：最後 14 天當測試
train, test = daily.iloc[:-14], daily.iloc[-14:]
print(f"\n訓練集 {len(train)} 天，測試集 {len(test)} 天")

# SARIMA(1,1,1)(1,1,0,7) -- 週期 = 7 天
from statsmodels.tsa.statespace.sarimax import SARIMAX
model_sarima = SARIMAX(
    train, order=(1, 1, 1), seasonal_order=(1, 1, 0, 7),
).fit(disp=False)
forecast_sarima = model_sarima.forecast(steps=14)
mae_sarima = mean_absolute_error(test.values, forecast_sarima.values)
print(f"\nSARIMA(1,1,1)(1,1,0,7)：MAE={mae_sarima:.3f}，AIC={model_sarima.aic:.2f}")

# 基準：訓練集最後 7 日滾動平均（視為未來 14 天的常數預測）
baseline_val = train.rolling(7).mean().iloc[-1]
baseline_pred = pd.Series(baseline_val, index=test.index)
mae_baseline = mean_absolute_error(test.values, baseline_pred.values)
print(f"7 日滾動平均基準：MAE={mae_baseline:.3f}")

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(train.index[-30:], train.values[-30:], color="#6B6B6B",
        linewidth=1.2, label="訓練（最後 30 天）")
ax.plot(test.index, test.values, color="#1A1A1A", linewidth=2,
        marker="o", markersize=4, label="實際")
ax.plot(test.index, forecast_sarima.values, color="#D97757", linewidth=1.8,
        marker="^", markersize=4, linestyle="--",
        label=f"SARIMA (MAE={mae_sarima:.2f})")
ax.plot(test.index, baseline_pred.values, color="#6A9BCC", linewidth=1.8,
        linestyle=":", label=f"7 日滾動平均 (MAE={mae_baseline:.2f})")
ax.set_title("流感每日通報數：SARIMA vs 滾動平均基準", fontweight="bold")
ax.set_xlabel("日期（Date）")
ax.set_ylabel("每日通報數")
ax.legend()
ax.set_ylim(bottom=0)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
fig.autofmt_xdate()
plt.tight_layout()
plt.show()

if mae_sarima < mae_baseline:
    print("\n→ SARIMA 優於滾動平均基準：因為 SARIMA 額外捕捉了週末通報下降的 7 天週期")
else:
    print("\n→ 本次滾動平均基準表現接近或優於 SARIMA，顯示資料量不大時簡單基準仍具參考價值")
print("→ 無論哪個模型較準，兩者都需要 >= 1 個完整週期的資料才能學到週間型態")

## 題目 6 解答

In [ ]:
import numpy as np

# --- 資料：COVID-19 90 天通報 line list ---
rng = np.random.default_rng(719)
start = pd.Timestamp("2026-03-01")
n_days = 90
dates_all = pd.date_range(start, periods=n_days, freq="D")
day_idx = np.arange(n_days)

# 單一波流行：先升後降，高峰約在第 40 天，並疊加「週末通報較少」的雜訊
wave_shape = np.exp(-0.5 * ((day_idx - 40) / 14) ** 2)
dow_weight = np.where(dates_all.dayofweek < 5, 1.0, 0.5)
weights = (0.05 + wave_shape) * dow_weight
probs = weights / weights.sum()

n_cases = 600
idx = rng.choice(n_days, size=n_cases, p=probs)
line_list = pd.DataFrame({
    "case_id": np.arange(1, n_cases + 1),
    "report_date": dates_all[idx],
}).sort_values("report_date").reset_index(drop=True)

daily = line_list.groupby("report_date").size()
daily = daily.asfreq("D", fill_value=0)
daily.name = "cases"
print(f"序列長度：{len(daily)} 天 | 總確診數：{daily.sum()}")

# 滾動平均窗口比較
print("\n=== 滾動平均窗口 MAE 比較 ===")
best_w, best_mae = 3, float("inf")
for w in [3, 7, 14]:
    pred_w = daily.rolling(window=w).mean().shift(1).dropna()
    actual_w = daily.loc[pred_w.index]
    mae_w = mean_absolute_error(actual_w, pred_w)
    print(f"  window={w:>2d}  MAE={mae_w:.3f}")
    if mae_w < best_mae:
        best_w, best_mae = w, mae_w
print(f"\n→ 最佳窗口：window={best_w}（MAE={best_mae:.3f}）")

# 高峰日比較
roll7 = daily.rolling(7, min_periods=1).mean()
peak_raw = daily.idxmax()
peak_smooth = roll7.idxmax()
print(f"\n原始每日數高峰日：{peak_raw.date()}（{daily.max()} 例)")
print(f"7 日滾動平均高峰日：{peak_smooth.date()}（{roll7.max():.1f} 例)")

fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(daily.index, daily.values, width=1.0, color="#6A9BCC",
       edgecolor="white", alpha=0.6, label="每日新增")
ax.plot(roll7.index, roll7.values, color="#D97757", linewidth=2,
        label="7 日滾動平均")
ax.axvline(peak_raw, color="#6A9BCC", linestyle=":", linewidth=1.5)
ax.axvline(peak_smooth, color="#D97757", linestyle=":", linewidth=1.5)
ax.set_title("COVID-19 每日新增與 7 日滾動平均", fontweight="bold")
ax.set_xlabel("通報日期（Report Date）")
ax.set_ylabel("每日新增病例數")
ax.legend()
ax.set_ylim(bottom=0)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
fig.autofmt_xdate()
plt.tight_layout()
plt.show()

print("\n→ 原始每日新增數受「週末通報延遲」影響而上下震盪，容易誤判高峰或轉折點")
print("→ 滾動平均能平滑雜訊、呈現真實趨勢，是公衛監測儀表板最常用的呈現方式")

## 題目 7 解答

In [ ]:
import numpy as np
import statsmodels.api as sm
import statsmodels.formula.api as smf

# --- 資料：登革熱（Dengue）一年通報 line list ---
rng = np.random.default_rng(709)
start = pd.Timestamp("2025-01-01")
n_days = 365
dates_all = pd.date_range(start, periods=n_days, freq="D")
doy = np.arange(n_days)

# 夏季高峰：約 6-9 月（病媒蚊密度隨氣溫濕度上升）
summer_shape = np.exp(-0.5 * ((doy - 210) / 35) ** 2)
weights = 0.05 + summer_shape
probs = weights / weights.sum()

n_cases = 520
idx = rng.choice(n_days, size=n_cases, p=probs)
line_list = pd.DataFrame({
    "case_id": np.arange(1, n_cases + 1),
    "report_date": dates_all[idx],
    "township": rng.choice(["A區", "B區", "C區"], size=n_cases, p=[0.5, 0.3, 0.2]),
}).sort_values("report_date").reset_index(drop=True)

daily = line_list.groupby("report_date").size()
daily = daily.asfreq("D", fill_value=0)
daily.name = "cases"
print(f"序列長度：{len(daily)} 天 | 總通報數：{daily.sum()}")

# lag 特徵
ts = daily.to_frame("cases").reset_index(names="date")
ts["day_idx"] = range(len(ts))
ts["lag_1"] = ts["cases"].shift(1)
ts["lag_2"] = ts["cases"].shift(2)
ts_model = ts.dropna().reset_index(drop=True)
print(f"可用列數：{len(ts_model)}")

# dispersion ratio
disp = ts_model["cases"].var() / ts_model["cases"].mean()
print(f"\ndispersion = variance / mean = {disp:.2f}")
print("→ > 1.5 視為過度離散 → 改用 Negative Binomial\n")

# Poisson 迴歸
model_pois = smf.glm(
    "cases ~ lag_1 + lag_2 + day_idx", data=ts_model, family=sm.families.Poisson(),
).fit()
pred_pois = model_pois.predict(ts_model)
mae_pois = mean_absolute_error(ts_model["cases"], pred_pois)
print(f"Poisson + lag：MAE={mae_pois:.3f}，AIC={model_pois.aic:.2f}")

# Negative Binomial 迴歸
model_nb = smf.glm(
    "cases ~ lag_1 + lag_2 + day_idx", data=ts_model,
    family=sm.families.NegativeBinomial(alpha=1.0),
).fit()
pred_nb = model_nb.predict(ts_model)
mae_nb = mean_absolute_error(ts_model["cases"], pred_nb)
print(f"Negative Binomial + lag：MAE={mae_nb:.3f}，AIC={model_nb.aic:.2f}")

# IRR
irr_table = pd.DataFrame({
    "coef (log scale)": model_nb.params,
    "IRR exp(coef)": np.exp(model_nb.params),
})
print("\n=== Negative Binomial 係數表（IRR）===")
print(irr_table.round(3))

lag1_irr = np.exp(model_nb.params["lag_1"])
print(f"\n→ lag_1 的 IRR = {lag1_irr:.3f}：前一天每多 1 例，隔天預期通報數變為 {lag1_irr:.2f} 倍")
print(f"→ dispersion={disp:.2f} 明顯 > 1.5，Negative Binomial 的 AIC（{model_nb.aic:.1f}）")
print(f"  低於 Poisson（{model_pois.aic:.1f}），代表 NB 更適合這種群聚、過度離散的病媒傳播資料")
print("→ 登革熱夏季高峰與氣溫、降雨提升病媒蚊（埃及斑蚊/白線斑蚊）密度及叮咬頻率有關")

## 題目 8 解答

In [ ]:
import numpy as np
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.tsa.arima.model import ARIMA

# --- 資料：松柏護理之家「二次污染」延伸情境（含第一波 + 平靜期 + 第二波）---
rng = np.random.default_rng(713)
start = pd.Timestamp("2026-02-01")

# 第一波：與主資料集類似的 21 天 outbreak，高峰約在第 8 天
wave1_days = 21
wave1_shape = np.exp(-0.5 * ((np.arange(wave1_days) - 8) / 3.5) ** 2)
wave1_dates = pd.date_range(start, periods=wave1_days, freq="D")

# 第二波：55 天後熱水系統再次污染，規模較小、較集中
wave2_start = start + pd.Timedelta(days=55)
wave2_days = 15
wave2_shape = np.exp(-0.5 * ((np.arange(wave2_days) - 6) / 3) ** 2)
wave2_dates = pd.date_range(wave2_start, periods=wave2_days, freq="D")

def _sample_wave(dates_wave, shape, n, rng):
    probs = shape / shape.sum()
    idx = rng.choice(len(dates_wave), size=n, p=probs)
    return dates_wave[idx]

onset1 = _sample_wave(wave1_dates, wave1_shape, 92, rng)
onset2 = _sample_wave(wave2_dates, wave2_shape, 34, rng)
onset_all = np.concatenate([onset1, onset2])

line_list = pd.DataFrame({
    "case_id": np.arange(1, len(onset_all) + 1),
    "symptom_onset_date": onset_all,
}).sort_values("symptom_onset_date").reset_index(drop=True)
print(f"總病例數（兩波合計）：{len(line_list)}")

# 完整每日發病數序列
full_range = pd.date_range(wave1_dates.min(), wave2_dates.max(), freq="D")
daily = line_list.groupby("symptom_onset_date").size()
daily = daily.reindex(full_range, fill_value=0)
daily.name = "cases"
print(f"序列長度：{len(daily)} 天（{daily.index.min().date()} ~ {daily.index.max().date()}）")

# 訓練集 = 第一波 + 平靜期（前 51 天）；測試集 = 第 52 天之後（涵蓋第二波）
cutoff = wave1_days + 30
train, test = daily.iloc[:cutoff], daily.iloc[cutoff:]
print(f"訓練集 {len(train)} 天，測試集 {len(test)} 天（第二波涵蓋於測試集內）")

# (a) 3 日滾動平均，取訓練集最後一個值當作未來的固定預測
roll_val = train.rolling(3, min_periods=1).mean().iloc[-1]
pred_roll = pd.Series(roll_val, index=test.index)
mae_roll = mean_absolute_error(test.values, pred_roll.values)

# (b) Poisson 迴歸 + lag，迭代預測（用前一步的預測值當作下一步的 lag）
ts = train.to_frame("cases").reset_index(names="date")
ts["day_idx"] = range(len(ts))
ts["lag_1"] = ts["cases"].shift(1)
ts["lag_2"] = ts["cases"].shift(2)
ts_model = ts.dropna().reset_index(drop=True)
model_pois = smf.glm(
    "cases ~ lag_1 + lag_2 + day_idx", data=ts_model, family=sm.families.Poisson(),
).fit()

history = list(train.values[-2:])
pred_pois = []
for i in range(len(test)):
    row = pd.DataFrame({
        "lag_1": [history[-1]], "lag_2": [history[-2]], "day_idx": [len(train) + i],
    })
    p = model_pois.predict(row).iloc[0]
    pred_pois.append(p)
    history.append(p)
mae_pois = mean_absolute_error(test.values, pred_pois)

# (c) ARIMA(1,1,1)
model_arima = ARIMA(train, order=(1, 1, 1)).fit()
forecast_arima = model_arima.forecast(steps=len(test))
mae_arima = mean_absolute_error(test.values, forecast_arima.values)

print("\n=== 三模型 MAE 比較（測試期涵蓋第二波）===")
print(f"  (a) 3 日滾動平均（固定值）  MAE={mae_roll:.3f}")
print(f"  (b) Poisson + lag（迭代預測） MAE={mae_pois:.3f}")
print(f"  (c) ARIMA(1,1,1)             MAE={mae_arima:.3f}")

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(train.index, train.values, color="#6B6B6B", linewidth=1.2, label="訓練期（第一波+平靜期）")
ax.plot(test.index, test.values, color="#1A1A1A", linewidth=2,
        marker="o", markersize=4, label="實際（含第二波）")
ax.plot(test.index, pred_roll.values, color="#6A9BCC", linestyle=":", linewidth=1.8,
        label=f"3 日滾動平均 (MAE={mae_roll:.2f})")
ax.plot(test.index, pred_pois, color="#788C5D", linestyle="--", linewidth=1.8,
        label=f"Poisson + lag (MAE={mae_pois:.2f})")
ax.plot(test.index, forecast_arima.values, color="#D97757", linestyle="--", linewidth=1.8,
        label=f"ARIMA(1,1,1) (MAE={mae_arima:.2f})")
ax.set_title("松柏護理之家二次污染：三模型預測 vs 實際發病數", fontweight="bold")
ax.set_xlabel("發病日期（Date of Symptom Onset）")
ax.set_ylabel("每日發病數")
ax.legend(fontsize=8)
ax.set_ylim(bottom=0)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
fig.autofmt_xdate()
plt.tight_layout()
plt.show()

print("\n→ 三個模型在測試期幾乎都預測接近 0，完全沒有預見第二波的出現")
print("→ 因為三個模型都只從『第一波 + 平靜期』的歷史型態外推，")
print("  而第二波是一個全新的暴露事件（熱水系統再次污染），不是既有趨勢的延續")
print("→ 啟示：統計模型只能『外推過去的型態』，無法預測全新的暴露源；")
print("  即時監測系統仍需搭配環境採檢、水質監測等主動措施，才能及早偵測二次污染")